# Phase 1: High-Fidelity Physics Data Generation
**Stabford Code In Place — Project: LNO-IonTransport Pipeline**

### Overview & Theory:
This notebook establishes the foundational dataset for our operator learning models. The transport of ionic species under thermal fluctuations is governed by the stochastic **Poisson-Nernst-Planck (PNP)** system. We simulate these electrodiffusive fields across 5 distinct thermodynamic regimes:
1. **Low Noise:** Clean deterministic transport.
2. **Stochastic:** Thermal fluctuations dominated regime.
3. **Heavy Dissipation:** Open-system boundary damping.
4. **Collapse:** High gradient electrochemical breakdown.
5. **Metastable:** Dynamic multi-attractor state.

The output will create raw physics tensors containing state snapshots ($u_t, u_{t+1}$), electrostatic potential ($\phi$), ionic flux, and noise fields.

In [3]:
import os
import sys
import numpy as np
import pandas as pd

# Setup foundational paths
sys.path.append(os.path.abspath(os.path.join('..')))

# Initialize directory structure for data saving
directories = ["data", "dataset/train", "dataset/val", "metadata"]
for d in directories:
    os.makedirs(d, exist_ok=True)

print("[+] Phase 1 Environment Ready. Storage paths initialized.")

[+] Phase 1 Environment Ready. Storage paths initialized.


In [2]:
!mkdir -p data dataset metadata

In [1]:
%%writefile config.yaml
# Master Configuration for Lindblad Neural Operator (LNO) Pipeline

infrastructure:
  seed: 42
  device: "cuda" # Automatically falls back to cpu if cuda unavailable
  workspace: "LNO-IonTransport"

physics:
  nx: 128                # Spatial grid points
  nt: 100                # Temporal tracking steps per trajectory
  dx: 0.01               # Grid resolution
  dt: 0.001              # Time step interval
  D: 0.1                 # Diffusion coefficient baseline
  mu: 1.0                # Ionic mobility factor
  epsilon: 0.05          # Membrane permittivity constant

dataset:
  raw_dir: "data/raw"
  processed_dir: "data/processed"
  splits_dir: "dataset"
  trajectories_per_regime: 60
  train_ratio: 0.80
  val_ratio: 0.10
  test_ratio: 0.10

model:
  name: "LindbladNeuralOperator"
  in_channels: 4         # Inputs: c(x,t), phi(x,t), noise(x,t), gamma
  out_channels: 1        # Target: c(x,t+1)
  modes: 16              # Number of Fourier modes to keep
  width: 64              # Hidden dimension / lift channels width
  num_layers: 4          # Depth of spectral operator layers

training:
  epochs: 150
  batch_size: 32
  learning_rate: 0.001
  weight_decay: 0.0001
  scheduler:
    step_size: 30
    gamma: 0.5
  loss_weights:
    data_l2: 1.0
    physics_pnp: 0.1     # Physics-informed loss weight constraint

Writing config.yaml


In [ ]:
%%writefile data/__init__.py
"""
LNO-IonTransport Production Pipeline: Data Module Initialization
"""
from .pnp_simulator import PhysicsOperatorSimulator
from .dataset_loader import IonTransportDataset

__all__ = ["PhysicsOperatorSimulator", "IonTransportDataset"]

Writing data/__init__.py


In [ ]:
%%writefile data/stochastic_forcing.py
"""
Stochastic Forcing Layer: Implements non-trivial microscopic fluctuations
representing realistic nanoscale thermal perturbations.
"""

import numpy as np

def generate_gaussian_noise(nx: int, sigma: float = 0.1) -> np.ndarray:
    """Standard spatial white noise wrapper."""
    return sigma * np.random.randn(nx)

def ornstein_uhlenbeck_noise(nx: int, theta: float = 0.2, sigma: float = 0.1, dt: float = 0.001) -> np.ndarray:
    """
    Simulates a continuous temporal Ornstein-Uhlenbeck stochastic relaxation track:
    deta = -theta * eta * dt + sigma * dW_t
    """
    eta = np.zeros(nx)
    spatial_wiener_increment = np.random.randn(nx)

    # Solve via Euler-Maruyama discretization scheme
    for i in range(1, nx):
        eta[i] = eta[i-1] - theta * eta[i-1] * dt + sigma * np.sqrt(dt) * spatial_wiener_increment[i]

    return eta

def generate_correlated_noise(nx: int, sigma: float = 0.1, correlation_scale: int = 5) -> np.ndarray:
    """Applies a Gaussian kernel smooth convolution to simulate localized spatial correlations."""
    raw_noise = np.random.randn(nx)
    grid_space = np.linspace(-2, 2, correlation_scale)
    kernel = np.exp(-grid_space ** 2)
    kernel /= kernel.sum()

    correlated = np.convolve(raw_noise, kernel, mode="same")
    return sigma * correlated

Writing data/stochastic_forcing.py


In [ ]:
%%writefile data/boundary_conditions.py
"""
Membrane & Domain Physics Boundary Condition Handlers.
Guarantees mathematical boundary validity across transport loops.
"""

import numpy as np

def apply_dirichlet_boundary(u: np.ndarray, left_val: float = 1.0, right_val: float = 0.1) -> np.ndarray:
    """Forces fixed concentrations or potentials at the system edges."""
    u[0] = left_val
    u[-1] = right_val
    return u

def apply_neumann_boundary(u: np.ndarray) -> np.ndarray:
    """Implements zero-flux / zero-gradient insulating physical barriers."""
    u[0] = u[1]
    u[-1] = u[-2]
    return u

def apply_periodic_boundary(u: np.ndarray) -> np.ndarray:
    """Wraps field states into continuous torus layouts."""
    u[0] = u[-2]
    u[-1] = u[1]
    return u

def apply_reflective_boundary(u: np.ndarray) -> np.ndarray:
    """Perfect particle mirroring behavior matching physical system boxes."""
    u[0] = u[1]
    u[-1] = u[-2]
    return u

Writing data/boundary_conditions.py


In [ ]:
%%writefile data/transport_metrics.py
"""
Physics Analysis Module: Computes high-fidelity statistical signatures,
entropy tracking, and local instability spatial breakdowns.
"""

import numpy as np

def compute_entropy(c: np.ndarray, dx: float) -> float:
    """
    Computes global Shannon Transport Entropy: S(t) = - ∫ c * ln(c) dx
    Provides continuous non-equilibrium tracking profiles.
    """
    c_safe = np.clip(c, 1e-8, None)
    entropy_value = -np.sum(c_safe * np.log(c_safe)) * dx
    return float(entropy_value)

def compute_flux_energy(flux: np.ndarray) -> float:
    """Computes integrated transport energy variance."""
    return float(np.mean(flux ** 2))

def compute_dissipation_rate(D: float, grad_c: np.ndarray) -> float:
    """Computes spatial thermodynamic energy loss profile means."""
    return float(D * np.mean(grad_c ** 2))

def compute_instability(flux: np.ndarray) -> np.ndarray:
    """
    Generates sharp localized gradients monitoring structural transport breakdown:
    instability = |grad(Flux)|
    """
    return np.abs(np.gradient(flux))

def compute_mass_error(c0: np.ndarray, c1: np.ndarray) -> float:
    """Evaluates total mass retention discrepancy over a system step."""
    return float(np.abs(np.sum(c0) - np.sum(c1)))

Writing data/transport_metrics.py


In [ ]:
%%writefile data/regime_configs.py
"""
Multi-Regime Physics Configurations Registry.
Enforces modular reproducible testing profiles across distinct physical zones.
"""

REGIMES = {
    "low_noise": {
        "noise": 0.02,
        "gamma": 0.01,
        "description": "Clean classical continuous deterministic electrodiffusive transport"
    },
    "stochastic": {
        "noise": 0.45,
        "gamma": 0.05,
        "description": "Thermal fluctuations dominated active molecular ion transport"
    },
    "heavy_dissipation": {
        "noise": 0.15,
        "gamma": 0.35,
        "description": "Strong open-system environmental coupling and damping activation"
    },
    "collapse": {
        "noise": 0.65,
        "gamma": 0.50,
        "description": "High gradient critical system breakdowns and bottleneck localizations"
    },
    "metastable": {
        "noise": 0.30,
        "gamma": 0.15,
        "description": "Dynamic multi-attractor state hopping under non-equilibrium states"
    }
}

Writing data/regime_configs.py


In [ ]:
%%writefile data/preprocessing.py
"""
Data Normalization & Engineering Preprocessing Pipeline components.
"""

import numpy as np

def normalize(x: np.ndarray) -> np.ndarray:
    """Standardizes input fields by tracking mean and variance profiles."""
    mean = np.mean(x)
    std = np.std(x) + 1e-8
    return (x - mean) / std

def clip_outliers(x: np.ndarray, threshold: float = 5.0) -> np.ndarray:
    """Caps extreme signal bursts preventing mathematical training overflow spikes."""
    return np.clip(x, -threshold, threshold)

def spectral_normalize(x: np.ndarray) -> np.ndarray:
    """Scales representation coefficients across frequency axes uniformly."""
    fft_vals = np.fft.fft(x)
    fft_vals /= (np.max(np.abs(fft_vals)) + 1e-8)
    return np.real(np.fft.ifft(fft_vals))

Writing data/preprocessing.py


In [ ]:
%%writefile data/validation.py
"""
Physics Consistency Assurance Framework: Assures simulations respect system invariants.
"""

import numpy as np

def check_nan(x: np.ndarray) -> bool:
    return int(np.isnan(x).sum()) == 0

def check_inf(x: np.ndarray) -> bool:
    return int(np.isinf(x).sum()) == 0

def check_mass_conservation(c0: np.ndarray, c1: np.ndarray, tolerance: float = 1e-2) -> bool:
    m0 = np.sum(c0)
    m1 = np.sum(c1)
    return float(np.abs(m0 - m1)) < tolerance

def check_positive_density(c: np.ndarray) -> bool:
    return bool(np.all(c >= -1e-7))

def validate_sample(c0: np.ndarray, c1: np.ndarray) -> bool:
    """Verifies compliance metrics across raw field configurations."""
    return (
        check_nan(c0) and check_nan(c1) and
        check_inf(c0) and check_inf(c1) and
        check_positive_density(c0) and check_positive_density(c1)
    )

def dataset_statistics(x: np.ndarray) -> dict:
    """Computes execution metadata blocks."""
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x))
    }

Writing data/validation.py


In [ ]:
%%writefile data/dataset_loader.py

"""
PyTorch Structured Data Pipeline Loader.
Stacks input arrays matching LNO config: [state_t, phi, flux, noise, dissipation, instability]
"""

import numpy as np
import torch
from torch.utils.data import Dataset

class IonTransportDataset(Dataset):
    def __init__(self, root: str):
        """Loads physical array configurations saved across processing splits."""
        self.state_t = np.load(f"{root}/state_t.npy")
        self.state_t1 = np.load(f"{root}/state_t1.npy")
        self.phi = np.load(f"{root}/phi.npy")
        self.flux = np.load(f"{root}/flux.npy")
        self.noise = np.load(f"{root}/noise.npy")
        self.dissipation = np.load(f"{root}/dissipation.npy")
        self.entropy = np.load(f"{root}/entropy.npy")
        self.instability = np.load(f"{root}/instability.npy")

    def __len__(self) -> int:
        return len(self.state_t)

    def __getitem__(self, idx: int):
        # Multichannel structural stacking along axis=0 matching operator input dimension layouts
        x = np.stack([
            self.state_t[idx],
            self.phi[idx],
            self.flux[idx],
            self.noise[idx],
            self.dissipation[idx],
            self.instability[idx]
        ], axis=0)

        y = self.state_t1[idx]

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )


Overwriting data/dataset_loader.py


In [ ]:
%%writefile data/pnp_simulator.py
"""
Stochastic Poisson-Nernst-Planck Coupled PDE Engine.
Solves nonlinear electrodiffusive profiles with custom open system spatial dissipative operators.
"""

import numpy as np
from .stochastic_forcing import ornstein_uhlenbeck_noise
from .boundary_conditions import apply_dirichlet_boundary, apply_reflective_boundary
from .transport_metrics import compute_entropy, compute_instability

class PhysicsOperatorSimulator:
    def __init__(self, config: dict):
        self.nx = config["physics"]["nx"]
        self.nt = config["physics"]["nt"]
        self.dx = config["physics"]["dx"]
        self.dt = config["physics"]["dt"]
        self.D = config["physics"]["D"]
        self.mu = config["physics"]["mu"]
        self.epsilon = config["physics"]["epsilon"]

    def _solve_poisson(self, c: np.ndarray) -> np.ndarray:
        """Solves electric potentials via Finite Difference matrix structures (-eps * d2Phi/dx2 = z * e * c)"""
        A = np.zeros((self.nx, self.nx))
        b = np.zeros(self.nx)

        # Build tridiagonal finite-difference matrix layouts
        for i in range(1, self.nx - 1):
            A[i, i - 1] = 1.0 / (self.dx ** 2)
            A[i, i] = -2.0 / (self.dx ** 2)
            A[i, i + 1] = 1.0 / (self.dx ** 2)
            b[i] = -c[i] / self.epsilon

        # Dirichlet boundary limits tracking potential drops
        A[0, 0] = 1.0; b[0] = 0.5
        A[-1, -1] = 1.0; b[-1] = 0.0

        return np.linalg.solve(A, b)

    def compute_trajectory(self, noise_sigma: float, gamma: float) -> tuple:
        """Propagates numerical state transformations recording specialized thermodynamic signals."""
        states_t = []
        states_t1 = []
        phis_t = []
        fluxes_t = []
        noises_t = []
        dissipations_t = []
        gammas_t = []
        entropies_t = []
        instabilities_t = []

        # Initialize smooth concentration profile across spatial grid
        c = 1.0 + 0.5 * np.sin(2 * np.pi * np.linspace(0, 1, self.nx))
        c = apply_dirichlet_boundary(c, left_val=1.2, right_val=0.2)

        for t in range(self.nt):
            phi = self._solve_poisson(c)
            grad_c = np.gradient(c, self.dx)
            grad_phi = np.gradient(phi, self.dx)

            # Nernst-Planck ionic flux calculation
            flux = -self.D * grad_c - self.mu * c * grad_phi
            div_flux = np.gradient(flux, self.dx)

            # High-fidelity Ornstein-Uhlenbeck noise injection
            eta = ornstein_uhlenbeck_noise(self.nx, theta=0.2, sigma=noise_sigma, dt=self.dt)

            # Dissipation mapping: open system relaxation contraction
            local_dissipation = self.D * (grad_c ** 2)

            # Compute system updates
            c_next = c + self.dt * (-div_flux - gamma * local_dissipation + eta)
            c_next = np.clip(c_next, 1e-8, None) # Enforce positivity bound
            c_next = apply_dirichlet_boundary(c_next, left_val=1.2, right_val=0.2)

            # Log thermodynamics & tracking functions
            ent_val = compute_entropy(c, self.dx)
            inst_map = compute_instability(flux)

            # Store configurations
            states_t.append(c.copy())
            states_t1.append(c_next.copy())
            phis_t.append(phi.copy())
            fluxes_t.append(flux.copy())
            noises_t.append(eta.copy())
            dissipations_t.append(local_dissipation.copy())
            gammas_t.append(np.full(self.nx, gamma))
            entropies_t.append(np.array([ent_val]))
            instabilities_t.append(inst_map.copy())

            c = c_next.copy()

        return (
            np.array(states_t, dtype=np.float32),
            np.array(states_t1, dtype=np.float32),
            np.array(phis_t, dtype=np.float32),
            np.array(fluxes_t, dtype=np.float32),
            np.array(noises_t, dtype=np.float32),
            np.array(dissipations_t, dtype=np.float32),
            np.array(gammas_t, dtype=np.float32),
            np.array(entropies_t, dtype=np.float32),
            np.array(instabilities_t, dtype=np.float32)
        )

Writing data/pnp_simulator.py


In [ ]:
%%writefile data/dataset_generator.py
"""
Master Dataset Pipeline Execution Orchestrator.
Generates structural regimes, processes distributions, handles splits and saves HDF5/NPY.
"""

import os
import json
import numpy as np
import h5py
import yaml

from data.pnp_simulator import PhysicsOperatorSimulator
from data.regime_configs import REGIMES
from data.validation import validate_sample, dataset_statistics

def main():
    # Load pipeline parameters master configurations file
    with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)

    np.random.seed(config["infrastructure"]["seed"])
    simulator = PhysicsOperatorSimulator(config)
    trajectories_count = config["dataset"]["trajectories_per_regime"]

    all_state_t = []
    all_state_t1 = []
    all_phi = []
    all_flux = []
    all_noise = []
    all_diss = []
    all_gamma = []
    all_entropy = []
    all_instability = []

    print("=== Initiating Yuánlǐ AI High-Fidelity Physics Data Engine ===")
    for regime_name, parameters in REGIMES.items():
        print(f"Running Environment Phase Execution: [{regime_name}]")
        for _ in range(trajectories_count):
            st, st1, ph, fl, ns, ds, gm, ent, inst = simulator.compute_trajectory(
                noise_sigma=parameters["noise"],
                gamma=parameters["gamma"]
            )
            # Filter arrays through automated validation metrics checks
            if validate_sample(st[0], st1[0]):
                all_state_t.append(st)
                all_state_t1.append(st1)
                all_phi.append(ph)
                all_flux.append(fl)
                all_noise.append(ns)
                all_diss.append(ds)
                all_gamma.append(gm)
                all_entropy.append(ent)
                all_instability.append(inst)

    # Convert lists into continuous operational arrays
    X_state_t = np.concatenate(all_state_t, axis=0)
    Y_state_t1 = np.concatenate(all_state_t1, axis=0)
    X_phi = np.concatenate(all_phi, axis=0)
    X_flux = np.concatenate(all_flux, axis=0)
    X_noise = np.concatenate(all_noise, axis=0)
    X_diss = np.concatenate(all_diss, axis=0)
    X_gamma = np.concatenate(all_gamma, axis=0)
    X_entropy = np.concatenate(all_entropy, axis=0)
    X_instability = np.concatenate(all_instability, axis=0)

    # Perform unified array shuffling across coordinated blocks
    total_samples = len(X_state_t)
    indices = np.arange(total_samples)
    np.random.shuffle(indices)

    X_state_t = X_state_t[indices]
    Y_state_t1 = Y_state_t1[indices]
    X_phi = X_phi[indices]
    X_flux = X_flux[indices]
    X_noise = X_noise[indices]
    X_diss = X_diss[indices]
    X_gamma = X_gamma[indices]
    X_entropy = X_entropy[indices]
    X_instability = X_instability[indices]

    # Calculate train / val / test indices segment distributions
    train_end = int(config["dataset"]["train_ratio"] * total_samples)
    val_end = train_end + int(config["dataset"]["val_ratio"] * total_samples)

    splits = {
        "train": (0, train_end),
        "val": (train_end, val_end),
        "test": (val_end, total_samples)
    }

    # Save target specific arrays across layout paths
    for split_name, (start, end) in splits.items():
        split_path = f"dataset/{split_name}"
        os.makedirs(split_path, exist_ok=True)

        np.save(f"{split_path}/state_t.npy", X_state_t[start:end])
        np.save(f"{split_path}/state_t1.npy", Y_state_t1[start:end])
        np.save(f"{split_path}/phi.npy", X_phi[start:end])
        np.save(f"{split_path}/flux.npy", X_flux[start:end])
        np.save(f"{split_path}/noise.npy", X_noise[start:end])
        np.save(f"{split_path}/dissipation.npy", X_diss[start:end])
        np.save(f"{split_path}/gamma.npy", X_gamma[start:end])
        np.save(f"{split_path}/entropy.npy", X_entropy[start:end])
        np.save(f"{split_path}/instability.npy", X_instability[start:end])

    # Export high-end academic HDF5 unified file storage
    print("Exporting production-scale master HDF5 layout dataset...")
    h5_path = "dataset/operator_dataset.h5"
    with h5py.File(h5_path, "w") as hf:
        hf.create_dataset("state_t", data=X_state_t, compression="gzip")
        hf.create_dataset("state_t1", data=Y_state_t1, compression="gzip")
        hf.create_dataset("phi", data=X_phi, compression="gzip")
        hf.create_dataset("flux", data=X_flux, compression="gzip")
        hf.create_dataset("noise", data=X_noise, compression="gzip")
        hf.create_dataset("dissipation", data=X_diss, compression="gzip")
        hf.create_dataset("gamma", data=X_gamma, compression="gzip")
        hf.create_dataset("entropy", data=X_entropy, compression="gzip")
        hf.create_dataset("instability", data=X_instability, compression="gzip")

    # Generate metadata documents under metadata/ directories
    os.makedirs("metadata", exist_ok=True)
    os.makedirs("dataset/metadata", exist_ok=True)
    meta_path = "metadata"

    with open(f"{meta_path}/regimes.json", "w") as fj:
        json.dump(REGIMES, fj, indent=2)

    with open(f"{meta_path}/dataset_config.json", "w") as fj:
        json.dump(config["dataset"], fj, indent=2)

    stats = {
        "total_samples": total_samples,
        "density_stats": dataset_statistics(X_state_t),
        "flux_stats": dataset_statistics(X_flux),
        "entropy_range": dataset_statistics(X_entropy)
    }
    with open(f"{meta_path}/statistics.json", "w") as fj:
        json.dump(stats, fj, indent=2)

    print(f"=== Process Completed. Data vectors compiled successfully inside dataset/ splits. Total samples: {total_samples} ===")

if __name__ == "__main__":
    main()

Overwriting data/dataset_generator.py


In [ ]:
!python -m data.dataset_generator

In [ ]:
%%writefile data/dataset_generator.py
"""
Master Dataset Pipeline Execution Orchestrator.
Generates structural regimes, processes distributions, handles splits and saves HDF5/NPY.
"""

import os
import json
import numpy as np
import h5py
import yaml

from data.pnp_simulator import PhysicsOperatorSimulator
from data.regime_configs import REGIMES
from data.validation import validate_sample, dataset_statistics

def main():
    # Load pipeline parameters master configurations file
    with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)

    np.random.seed(config["infrastructure"]["seed"])
    simulator = PhysicsOperatorSimulator(config)
    trajectories_count = config["dataset"]["trajectories_per_regime"]

    all_state_t = []
    all_state_t1 = []
    all_phi = []
    all_flux = []
    all_noise = []
    all_diss = []
    all_gamma = []
    all_entropy = []
    all_instability = []

    print("=== Initiating Yuánlǐ AI High-Fidelity Physics Data Engine ===")
    for regime_name, parameters in REGIMES.items():
        print(f"Running Environment Phase Execution: [{regime_name}]")
        for _ in range(trajectories_count):
            st, st1, ph, fl, ns, ds, gm, ent, inst = simulator.compute_trajectory(
                noise_sigma=parameters["noise"],
                gamma=parameters["gamma"]
            )
            # Filter arrays through automated validation metrics checks
            if validate_sample(st[0], st1[0]):
                all_state_t.append(st)
                all_state_t1.append(st1)
                all_phi.append(ph)
                all_flux.append(fl)
                all_noise.append(ns)
                all_diss.append(ds)
                all_gamma.append(gm)
                all_entropy.append(ent)
                all_instability.append(inst)

    # Convert lists into continuous operational arrays
    X_state_t = np.concatenate(all_state_t, axis=0)
    Y_state_t1 = np.concatenate(all_state_t1, axis=0)
    X_phi = np.concatenate(all_phi, axis=0)
    X_flux = np.concatenate(all_flux, axis=0)
    X_noise = np.concatenate(all_noise, axis=0)
    X_diss = np.concatenate(all_diss, axis=0)
    X_gamma = np.concatenate(all_gamma, axis=0)
    X_entropy = np.concatenate(all_entropy, axis=0)
    X_instability = np.concatenate(all_instability, axis=0)

    # Perform unified array shuffling across coordinated blocks
    total_samples = len(X_state_t)
    indices = np.arange(total_samples)
    np.random.shuffle(indices)

    X_state_t = X_state_t[indices]
    Y_state_t1 = Y_state_t1[indices]
    X_phi = X_phi[indices]
    X_flux = X_flux[indices]
    X_noise = X_noise[indices]
    X_diss = X_diss[indices]
    X_gamma = X_gamma[indices]
    X_entropy = X_entropy[indices]
    X_instability = X_instability[indices]

    # Calculate train / val / test indices segment distributions
    train_end = int(config["dataset"]["train_ratio"] * total_samples)
    val_end = train_end + int(config["dataset"]["val_ratio"] * total_samples)

    splits = {
        "train": (0, train_end),
        "val": (train_end, val_end),
        "test": (val_end, total_samples)
    }

    # Save target specific arrays across layout paths
    for split_name, (start, end) in splits.items():
        split_path = f"dataset/{split_name}"
        os.makedirs(split_path, exist_ok=True)

        np.save(f"{split_path}/state_t.npy", X_state_t[start:end])
        np.save(f"{split_path}/state_t1.npy", Y_state_t1[start:end])
        np.save(f"{split_path}/phi.npy", X_phi[start:end])
        np.save(f"{split_path}/flux.npy", X_flux[start:end])
        np.save(f"{split_path}/noise.npy", X_noise[start:end])
        np.save(f"{split_path}/dissipation.npy", X_diss[start:end])
        np.save(f"{split_path}/gamma.npy", X_gamma[start:end])
        np.save(f"{split_path}/entropy.npy", X_entropy[start:end])
        np.save(f"{split_path}/instability.npy", X_instability[start:end])

    # Export high-end academic HDF5 unified file storage
    print("Exporting production-scale master HDF5 layout dataset...")
    h5_path = "dataset/operator_dataset.h5"
    with h5py.File(h5_path, "w") as hf:
        hf.create_dataset("state_t", data=X_state_t, compression="gzip")
        hf.create_dataset("state_t1", data=Y_state_t1, compression="gzip")
        hf.create_dataset("phi", data=X_phi, compression="gzip")
        hf.create_dataset("flux", data=X_flux, compression="gzip")
        hf.create_dataset("noise", data=X_noise, compression="gzip")
        hf.create_dataset("dissipation", data=X_diss, compression="gzip")
        hf.create_dataset("gamma", data=X_gamma, compression="gzip")
        hf.create_dataset("entropy", data=X_entropy, compression="gzip")
        hf.create_dataset("instability", data=X_instability, compression="gzip")

    # Generate metadata documents under metadata/ directories
    os.makedirs("metadata", exist_ok=True)
    os.makedirs("dataset/metadata", exist_ok=True)
    meta_path = "metadata"

    with open(f"{meta_path}/regimes.json", "w") as fj:
        json.dump(REGIMES, fj, indent=2)

    with open(f"{meta_path}/dataset_config.json", "w") as fj:
        json.dump(config["dataset"], fj, indent=2)

    stats = {
        "total_samples": total_samples,
        "density_stats": dataset_statistics(X_state_t),
        "flux_stats": dataset_statistics(X_flux),
        "entropy_range": dataset_statistics(X_entropy)
    }
    with open(f"{meta_path}/statistics.json", "w") as fj:
        json.dump(stats, fj, indent=2)

    print(f"=== Process Completed. Data vectors compiled successfully inside dataset/ splits. Total samples: {total_samples} ===")

if __name__ == "__main__":
    main()

Overwriting data/dataset_generator.py


In [ ]:
# Re-run the dataset generator script
!python -m data.dataset_generator

=== Initiating Yuánlǐ AI High-Fidelity Physics Data Engine ===
Running Environment Phase Execution: [low_noise]
Running Environment Phase Execution: [stochastic]
Running Environment Phase Execution: [heavy_dissipation]
Running Environment Phase Execution: [collapse]
Running Environment Phase Execution: [metastable]
Exporting production-scale master HDF5 layout dataset...
=== Process Completed. Data vectors compiled successfully inside dataset/ splits. Total samples: 30000 ===


In [ ]:
# List the contents of the metadata directory
!ls -F metadata/

# List the contents of the dataset/train directory as an example
!ls -F dataset/train/

dataset_config.json  regimes.json  statistics.json
dissipation.npy  flux.npy   instability.npy  phi.npy	   state_t.npy
entropy.npy	 gamma.npy  noise.npy	     state_t1.npy
